# Read alignment basics: BWA + samtools

The single most common shared-workstation genomics task: given a reference genome and a pile
of sequencing reads, figure out where each read came from. We generate a small synthetic
reference and reads (no download needed, no real dataset required to demonstrate the
pipeline), then run the standard tools: `bwa` to align, `samtools` to sort/index/inspect.

## 1. Generate a synthetic reference genome and reads

Reads are 100bp substrings sampled from random positions in the reference, with a small
per-base error rate (simulating real sequencing errors) - a standard way to test an alignment
pipeline without a real dataset. Since every read genuinely comes from this exact reference,
we know in advance that alignment should succeed for essentially all of them.

In [ ]:
import random

random.seed(0)


def random_seq(n):
    return "".join(random.choice("ACGT") for _ in range(n))


def mutate(seq, error_rate=0.01):
    bases = "ACGT"
    out = []
    for c in seq:
        if random.random() < error_rate:
            out.append(random.choice([b for b in bases if b != c]))
        else:
            out.append(c)
    return "".join(out)


reference = random_seq(5000)

read_len, n_reads = 100, 200
reads = []
for i in range(n_reads):
    pos = random.randint(0, len(reference) - read_len)
    read = mutate(reference[pos:pos + read_len], error_rate=0.01)
    reads.append((f"read_{i}_pos{pos}", read))

with open("/work/ref.fa", "w") as f:
    f.write(">chr1_synthetic\n")
    for i in range(0, len(reference), 70):
        f.write(reference[i:i + 70] + "\n")

with open("/work/reads.fq", "w") as f:
    for name, seq in reads:
        f.write(f"@{name}\n{seq}\n+\n{'I' * len(seq)}\n")

print(f"reference: {len(reference)} bp, {n_reads} reads of {read_len} bp each")
print("wrote /work/ref.fa and /work/reads.fq")

## 2. Index the reference and align

`bwa index` builds the data structures BWA needs to search the reference quickly.
`bwa mem` (the standard algorithm for reads ≥ 70bp) aligns the reads against it, producing a
SAM file - each line records where (if anywhere) a read matched.

In [ ]:
%cd /work
!bwa index ref.fa
!bwa mem ref.fa reads.fq > aligned.sam

## 3. Sort, index, and inspect with samtools

SAM is plain text; BAM is its compressed, indexed binary form, which is what most downstream
tools expect. `samtools flagstat` summarizes how many reads mapped - since every read here was
sampled directly from the reference (with only a small error rate), essentially all of them
should map.

In [ ]:
!samtools sort -o aligned.sorted.bam aligned.sam
!samtools index aligned.sorted.bam
!samtools flagstat aligned.sorted.bam

Expect a mapping rate near 100% - if it's much lower, something about the reference/read
generation or the alignment step went wrong and is worth investigating before trusting the
rest of the pipeline on real data.

## Next steps

- Increase `error_rate` in the read simulation and watch the mapping rate degrade - a feel for
  how alignment tools handle sequencing noise.
- Swap in a real reference (e.g. a small bacterial genome) and real reads from a public dataset
  instead of synthetic data.
- Add `samtools mpileup` or a variant caller (e.g. `bcftools call`) on top of the sorted BAM to
  go from "aligned reads" to "variants" - the next step in most real pipelines.